<a href="https://colab.research.google.com/github/ardominguezm/golden-age-semantic-reconfiguration/blob/main/notebooks/paper1_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Paper 1 — Golden Age Semantic Reconfiguration

**Current stage: Phase 11B — Structural Representation Calibration**

## Paper objective and novelty guardrail

The paper asks whether the Renaissance→Baroque transition in Spanish Golden Age poetry is better characterized as **gradual semantic drift** or as a **structural reorganization of relations among poetic concepts**. The empirical object remains a dynamic concept↔concept semantic network reconstructed from independently dated poems.

Phase 11 showed that the preregistered conservative representation (within-line co-occurrence, raw pair support ≥2) was computationally valid but structurally too sparse for a defensible rewiring analysis. Phase 11B therefore performs a **blind structural calibration** using only alternatives already frozen in Phase 10:

- **A — line/support≥2:** Phase-11 benchmark;
- **B — line/support≥1:** preregistered sensitivity;
- **C — 5 retained-content-token sliding context/support≥1:** preregistered alternative context.

No semantic distance, change point, Renaissance/Baroque label, 1580/1605 fit, or historical interpretation is computed here. Candidate selection is based only on frozen structural-feasibility criteria. If no candidate passes, Phase 11B returns **NO REPRESENTATION SELECTED** rather than tuning after seeing results.

The central novelty remains the later decomposition of **lexical turnover** from **relational rewiring among persistent concepts**, with chronology uncertainty and raw-vs-author-balanced controls.


In [1]:
import sys, subprocess, hashlib, urllib.request, re, shutil, unicodedata, math
from pathlib import Path
from collections import Counter, defaultdict
from itertools import combinations
from functools import lru_cache
from difflib import SequenceMatcher
import numpy as np, pandas as pd
import xml.etree.ElementTree as ET

SPACY_VERSION='3.8.7'; MODEL_NAME='es_core_news_sm'; MODEL_VERSION='3.8.0'; NETWORKX_VERSION='3.4.2'
MODEL_URL='https://github.com/explosion/spacy-models/releases/download/es_core_news_sm-3.8.0/es_core_news_sm-3.8.0-py3-none-any.whl'
MODEL_SHA256='e451a83d6df79b87e9eed0cb553f03e99e36a3bab18a7b79f0dcfd1fdf875e12'
wheel=Path('/content/es_core_news_sm-3.8.0-py3-none-any.whl')
if not wheel.exists() or hashlib.sha256(wheel.read_bytes()).hexdigest()!=MODEL_SHA256:
    urllib.request.urlretrieve(MODEL_URL,wheel)
assert hashlib.sha256(wheel.read_bytes()).hexdigest()==MODEL_SHA256
subprocess.run([sys.executable,'-m','pip','install','-q',f'spacy=={SPACY_VERSION}',f'networkx=={NETWORKX_VERSION}',str(wheel)],check=True)
import spacy, networkx as nx
assert spacy.__version__==SPACY_VERSION and nx.__version__==NETWORKX_VERSION
nlp=spacy.load(MODEL_NAME,disable=['parser','ner'])
assert nlp.meta.get('version')==MODEL_VERSION
print('Environment ready:',spacy.__version__,nlp.meta.get('version'),nx.__version__)


Environment ready: 3.8.7 3.8.0 3.4.2


In [2]:
SOURCES={
 'navarro_tei':('https://github.com/bncolorado/CorpusSonetosSigloDeOro.git','092a5fe70a4065a4d84bfed288bffd3851348f9c'),
 'gongora_scholarly':('https://github.com/gongoradigital/gongoraobra.git','3beadeecc059a7cc48499dc2683bb378a2630978'),
}
ROOT=Path('/content/gasr_phase11b_sources'); ROOT.mkdir(exist_ok=True)
def clone(name,url,commit):
    dst=ROOT/name
    if dst.exists(): shutil.rmtree(dst)
    subprocess.run(['git','clone','--quiet',url,str(dst)],check=True)
    subprocess.run(['git','-C',str(dst),'checkout','--quiet',commit],check=True)
    got=subprocess.check_output(['git','-C',str(dst),'rev-parse','HEAD'],text=True).strip()
    assert got==commit,(name,got,commit)
    return dst
paths={k:clone(k,*v) for k,v in SOURCES.items()}
N=paths['navarro_tei']; G=paths['gongora_scholarly']; XML_ID='{http://www.w3.org/XML/1998/namespace}id'
def local(tag): return tag.split('}')[-1] if '}' in tag else tag
def el_text(el): return '' if el is None else ' '.join(' '.join(el.itertext()).split())
def norm(s):
    s=unicodedata.normalize('NFKD',str(s)); s=''.join(c for c in s if not unicodedata.combining(c))
    return re.sub(r'[^a-z0-9]','',s.lower())
def years_1580_1626(s):
    return sorted(set(int(x) for x in re.findall(r'(?<!\d)(1[56]\d{2})(?!\d)',str(s)) if 1580<=int(x)<=1626))

rows=[]
for fp in sorted(N.rglob('*.xml')):
    root=ET.parse(fp).getroot(); lines=[el_text(x) for x in root.iter() if local(x.tag)=='l']; lines=[x for x in lines if x]
    if not lines: continue
    author=fp.parent.name; txt='\n'.join(lines)
    rows.append({'n_id':f'{author}::{fp.name}','author_dir':author,'source_file':str(fp.relative_to(N)),
                 'n_lines':len(lines),'lines':lines,'text_tei':txt,'signature':norm(txt),
                 'first2_signature':norm('\n'.join(lines[:2])),'first_line':lines[0]})
n=pd.DataFrame(rows); assert len(n)==5078,len(n)

primary_rows=[]
def add(pid,author,lo,hi,confidence,basis):
    primary_rows.append({'n_id':pid,'author_dir':author,'composition_min':int(lo),'composition_max':int(hi),
                         'temporal_confidence':confidence,'temporal_basis':basis})

groot=ET.parse(G/'gongora_obra-poetica.xml').getroot(); parent={child:par for par in groot.iter() for child in par}; grows=[]
for el in groot.iter():
    xid=el.attrib.get(XML_ID,'')
    if local(el.tag)!='div' or not xid.lower().startswith('poem'): continue
    lines=[el_text(x) for x in el.iter() if local(x.tag)=='l']; lines=[x for x in lines if x]
    if not lines: continue
    vals=[]; cur=el
    for _ in range(6):
        vals+=list(cur.attrib.values())
        if cur.text: vals.append(cur.text)
        for ch in list(cur):
            if local(ch.tag) in {'head','date','label'}: vals.append(el_text(ch))
            if ch.tail: vals.append(ch.tail)
        cur=parent.get(cur)
        if cur is None: break
    ys=sorted(set(y for v in vals for y in years_1580_1626(v))); txt='\n'.join(lines)
    grows.append({'g_id':xid,'n_lines':len(lines),'signature':norm(txt),'first2_signature':norm('\n'.join(lines[:2])),
                  'scholarly_year':ys[0] if len(ys)==1 else pd.NA,
                  'year_status':'unique' if len(ys)==1 else ('ambiguous' if len(ys)>1 else 'missing')})
g=pd.DataFrame(grows); g14=g[(g.n_lines==14)&g.signature.ne('')].copy(); g_by_id=g.set_index('g_id',drop=False)
ng=n[n.author_dir.eq('Gongora')].copy(); sig_to_gids=g14.groupby('signature').g_id.apply(list).to_dict()
links=[]
for r in ng.itertuples(index=False):
    exact_ids=sig_to_gids.get(r.signature,[])
    if len(exact_ids)==1: gid,score,method=exact_ids[0],1.0,'exact'
    else:
        best_gid,best_score=None,-1.0
        for gr in g14.itertuples(index=False):
            sc=SequenceMatcher(None,r.signature,gr.signature).ratio()
            if sc>best_score: best_gid,best_score=gr.g_id,sc
        gid,score,method=best_gid,best_score,'fuzzy'
    links.append({'n_id':r.n_id,'g_id':gid,'method':method,'score':float(score),'preaccept':method=='exact' or score>=0.98})
glink=pd.DataFrame(links); pre=glink[glink.preaccept].copy(); collisions=set(pre.g_id.value_counts()[lambda s:s>1].index)
glink['accept_phase4']=glink.preaccept&~glink.g_id.isin(collisions)
acc=glink[glink.accept_phase4].merge(g[['g_id','scholarly_year','year_status']],on='g_id',how='left')
acc=acc[acc.year_status.eq('unique')&acc.scholarly_year.notna()].copy()
for r in acc.itertuples(index=False):
    add(r.n_id,'Gongora',r.scholarly_year,r.scholarly_year,'A' if r.method=='exact' else 'B',
        'scholarly_chronology_year_exact_link' if r.method=='exact' else 'scholarly_chronology_year_fuzzy_link')
phase4_nids=set(acc.n_id); phase4_gids=set(acc.g_id); unmatched=ng[~ng.n_id.isin(phase4_nids)].copy()
first2_index=g14.groupby('first2_signature').g_id.apply(list).to_dict(); recovered=[]
for r in unmatched.itertuples(index=False):
    ids2=first2_index.get(r.first2_signature,[])
    if len(ids2)!=1: continue
    gid=ids2[0]
    if gid in phase4_gids: continue
    gr=g_by_id.loc[gid]; score=SequenceMatcher(None,r.signature,gr.signature).ratio()
    if score>=0.95 and gr.year_status=='unique' and pd.notna(gr.scholarly_year):
        recovered.append((r.n_id,gid,score,int(gr.scholarly_year)))
rec=pd.DataFrame(recovered,columns=['n_id','g_id','score','year']); dup=set(rec.g_id.value_counts()[lambda s:s>1].index) if len(rec) else set()
rec=rec[~rec.g_id.isin(dup)]
for r in rec.itertuples(index=False): add(r.n_id,'Gongora',r.year,r.year,'B','scholarly_chronology_year_variant_link')
assert sum(x['author_dir']=='Gongora' for x in primary_rows)==58

GAR={**{i:(1526,1532,'B','scholarly_phase_interval') for i in [1,2,3,4,6,26,27]},
     25:(1534,1535,'B','scholarly_interval'),33:(1535,1535,'A','historically_anchored_scholarly_year'),
     35:(1535,1535,'A','historically_anchored_scholarly_year'),
     **{i:(1533,1535,'B','revised_scholarly_interval') for i in [7,8,12,15,19,28,30,31]}}
for no,(lo,hi,conf,basis) in GAR.items(): add(f'GarcilasoDeLaVega::GarcilasoDeLaVega_{no:02d}.xml','GarcilasoDeLaVega',lo,hi,conf,basis)
for no,lo,hi,conf,basis in [(30,1596,1596,'B','Cadiz_1596'),(13,1598,1598,'A','FelipeII_tomb_1598'),(31,1597,1598,'B','Herrera_death_epitaph')]:
    add(f'Cervantes::Cervantes_{no}.xml','Cervantes',lo,hi,conf,basis)
for no,lo,hi,basis in [(224,1574,1574,'Alameda_CarlosV'),(279,1578,1579,'Barahona_Granada'),(276,1573,1574,'Bazan_Tunis'),
                       (300,1580,1582,'Portugal_to_H'),(281,1578,1578,'DonJuan_de_Austria')]:
    add(f'FernandoDeHerrera::FernandoDeHerrera_{no}.xml','FernandoDeHerrera',lo,hi,'B',basis)
for no in [2,19,4,5]: add(f'PedroEspinosa::PedroEspinosa_{no}.xml','PedroEspinosa',1594,1596,'B','Espinosa_happiness_period_1594_1596')
for no,year,basis in [(131,1609,'Carrillo_sonnet_1609'),(69,1611,'Aminta_1611'),(70,1611,'Aminta_1611'),(72,1611,'Aminta_1611'),
                      (76,1611,'Aminta_1611'),(42,1610,'HenryIV_1610'),(43,1610,'HenryIV_1610'),(45,1610,'HenryIV_1610'),(44,1624,'Osuna_1624')]:
    add(f'Quevedo::Quevedo_{no}.xml','Quevedo',year,year,'B',basis)

primary=pd.DataFrame(primary_rows).drop_duplicates('n_id').copy()
expected={'Gongora':58,'GarcilasoDeLaVega':18,'Quevedo':9,'FernandoDeHerrera':5,'PedroEspinosa':4,'Cervantes':3}
assert len(primary)==97 and primary.groupby('author_dir').size().to_dict()==expected
primary=primary.merge(n[['n_id','text_tei','lines','n_lines','source_file','first_line','signature']],on='n_id',how='left',validate='one_to_one')
assert primary.text_tei.notna().all()
print('FROZEN PRIMARY CHRONOLOGY REPRODUCED:',len(primary),'poems |',primary.author_dir.nunique(),'authors')


FROZEN PRIMARY CHRONOLOGY REPRODUCED: 97 poems | 6 authors


In [3]:
MAIN_POS={'NOUN','VERB','ADJ','ADV'}; MIN_LEMMA_LEN=2
line_records=[]
for r in primary.itertuples(index=False):
    for line_no,line in enumerate(r.lines,1): line_records.append((r.n_id,r.author_dir,line_no,line))
docs=list(nlp.pipe([x[3] for x in line_records],batch_size=128)); assert len(docs)==len(line_records)
token_rows=[]; global_order=0
for (pid,author,line_no,_),doc in zip(line_records,docs):
    for token_no,t in enumerate(doc):
        lemma=unicodedata.normalize('NFC',str(t.lemma_)).strip().lower(); pos=t.pos_
        keep=bool(t.is_alpha and pos in MAIN_POS and len(lemma)>=MIN_LEMMA_LEN)
        token_rows.append({'n_id':pid,'author_dir':author,'line_no':line_no,'token_no':token_no,'global_order':global_order,
                           'surface':t.text,'lemma':lemma,'pos':pos,'is_alpha':bool(t.is_alpha),'is_main_content':keep,
                           'concept':f'{lemma}::{pos}' if keep else pd.NA})
        global_order+=1
tokens=pd.DataFrame(token_rows); main_tok=tokens[tokens.is_main_content].copy()
concept_df=(main_tok[['n_id','concept','lemma','pos']].drop_duplicates(['n_id','concept'])
            .groupby(['concept','lemma','pos']).n_id.nunique().rename('poem_df').reset_index())
concept_tf=main_tok.groupby('concept').size().rename('token_frequency').reset_index()
vocab=concept_df.merge(concept_tf,on='concept',how='left')
MAIN_VOCAB=set(vocab.loc[vocab.poem_df.ge(2),'concept']); SENS_VOCAB=set(vocab.loc[vocab.poem_df.ge(3),'concept'])
assert len(MAIN_VOCAB)==668,len(MAIN_VOCAB)
assert len(SENS_VOCAB)==349,len(SENS_VOCAB)
assert len(tokens)==10439 and int(tokens.is_main_content.sum())==4358,(len(tokens),int(tokens.is_main_content.sum()))

poem_line_units={r.n_id:[set() for _ in range(r.n_lines)] for r in primary.itertuples(index=False)}
for (pid,line_no),grp in main_tok[main_tok.concept.isin(MAIN_VOCAB)].groupby(['n_id','line_no']):
    poem_line_units[pid][int(line_no)-1]=set(grp.concept)

TOKEN_WINDOW=5; poem_token_units={}
for pid,grp in main_tok[main_tok.concept.isin(MAIN_VOCAB)].sort_values('global_order').groupby('n_id',sort=False):
    seq=grp.concept.tolist()
    poem_token_units[pid]=[set(seq[i:i+TOKEN_WINDOW]) for i in range(max(0,len(seq)-TOKEN_WINDOW+1))]
assert set(poem_line_units)==set(primary.n_id)
assert set(poem_token_units)==set(primary.n_id)
assert all(len(v)>0 for v in poem_token_units.values())
print('Phase-10 representation reproduced exactly')
print('Tokens:',len(tokens),'| retained content:',int(tokens.is_main_content.sum()),'| main vocabulary:',len(MAIN_VOCAB))
print('Contexts ready: poetic line + 5 retained-content-token sliding windows')


Phase-10 representation reproduced exactly
Tokens: 10439 | retained content: 4358 | main vocabulary: 668
Contexts ready: poetic line + 5 retained-content-token sliding windows


In [4]:
CANDIDATES={
    'A_line_support2': {'context':'line','min_support':2,'selection_order':1,'role':'Phase-11 conservative benchmark'},
    'B_line_support1': {'context':'line','min_support':1,'selection_order':2,'role':'Phase-10 preregistered support sensitivity'},
    'C_token5_support1': {'context':'token5','min_support':1,'selection_order':3,'role':'Phase-10 preregistered context sensitivity'},
}
MIN_CONNECTED_FRACTION=1/3
MIN_GCC_CONNECTED_FRACTION=0.50
MIN_MEAN_DEGREE_CONNECTED=2.0
MIN_EFFECTIVE_EDGE_FRACTION=0.25
REQUIRED_PASS_CELLS=18

criteria=pd.DataFrame([
    ('connected_fraction_median',f'>={MIN_CONNECTED_FRACTION:.3f}','at least one-third of active concepts relationally embedded'),
    ('gcc_connected_fraction_median',f'>={MIN_GCC_CONNECTED_FRACTION:.2f}','majority component among connected concepts'),
    ('mean_degree_connected_median',f'>={MIN_MEAN_DEGREE_CONNECTED:.1f}','backbone richer than disjoint pairs'),
    ('effective_edge_fraction_median',f'>={MIN_EFFECTIVE_EDGE_FRACTION:.2f}','edge-weight mass not dominated by a tiny subset'),
    ('candidate pass',f'{REQUIRED_PASS_CELLS}/18 cells','all 9 windows in raw and author-balanced modes')
],columns=['criterion','threshold','interpretation'])
print('PHASE 11B STRUCTURAL CRITERIA FROZEN BEFORE B/C RESULTS')
display(criteria)


PHASE 11B STRUCTURAL CRITERIA FROZEN BEFORE B/C RESULTS


,criterion,threshold,interpretation
0,connected_fraction_median,>=0.333,at least one-third of active concepts relation...
1,gcc_connected_fraction_median,>=0.50,majority component among connected concepts
2,mean_degree_connected_median,>=2.0,backbone richer than disjoint pairs
3,effective_edge_fraction_median,>=0.25,edge-weight mass not dominated by a tiny subset
4,candidate pass,18/18 cells,all 9 windows in raw and author-balanced modes


In [5]:
primary_idx=primary.set_index('n_id',drop=False)
def effective_count(weights):
    w=np.asarray(weights,dtype=float); w=w[np.isfinite(w)&(w>0)]
    if len(w)==0: return 0.0
    p=w/w.sum()
    return float(np.exp(-(p*np.log(p)).sum()))
def context_dict(context):
    if context=='line': return poem_line_units
    if context=='token5': return poem_token_units
    raise ValueError(context)

@lru_cache(maxsize=None)
def build_metrics(ids_tuple,mode,context,min_support):
    ids=list(ids_tuple)
    if not ids: raise ValueError('Empty temporal window')
    units_by_poem=context_dict(context); sub=primary_idx.loc[ids]; author_counts=Counter(sub.author_dir)
    n_poems=len(ids); n_authors=len(author_counts); shares=np.array(list(author_counts.values()),dtype=float)/n_poems
    effective_authors=float(1/np.square(shares).sum()); top_author_share=float(shares.max())
    node_mass=defaultdict(float); pair_mass=defaultdict(float); raw_support=defaultdict(int); total_mass=0.0; total_units=0
    for pid in ids:
        units=units_by_poem[pid]; U=len(units)
        if U==0: continue
        author=primary_idx.loc[pid].author_dir
        unit_weight=1.0 if mode=='raw' else 1.0/(author_counts[author]*U)
        total_units+=U; total_mass+=U*unit_weight
        for concepts in units:
            concepts=sorted(concepts)
            for u in concepts: node_mass[u]+=unit_weight
            for u,v in combinations(concepts,2):
                pair_mass[(u,v)]+=unit_weight; raw_support[(u,v)]+=1
    active=sorted(u for u,c in node_mass.items() if c>0); G=nx.Graph(); G.add_nodes_from(active)
    for (u,v),support in raw_support.items():
        if support<int(min_support): continue
        pij=pair_mass[(u,v)]/total_mass; pi=node_mass[u]/total_mass; pj=node_mass[v]/total_mass
        if min(pij,pi,pj)<=0: continue
        ppmi=max(0.0,math.log2(pij/(pi*pj)))
        if ppmi>0: G.add_edge(u,v,weight=ppmi,raw_support=support,weighted_support=pair_mass[(u,v)])
    n_nodes=G.number_of_nodes(); n_edges=G.number_of_edges(); connected_nodes=[u for u,d in G.degree() if d>0]
    n_connected=len(connected_nodes); comps=list(nx.connected_components(G.subgraph(connected_nodes))) if n_connected else []
    gcc=max((len(c) for c in comps),default=0); weights=[d['weight'] for _,_,d in G.edges(data=True)]; eff_edges=effective_count(weights)
    return {'n_poems':n_poems,'n_authors':n_authors,'effective_authors':effective_authors,'top_author_share':top_author_share,
            'n_context_units':total_units,'context_mass':total_mass,'n_nodes':n_nodes,'n_connected_nodes':n_connected,
            'connected_fraction':n_connected/n_nodes if n_nodes else np.nan,'n_edges':n_edges,
            'density':nx.density(G) if n_nodes>1 else np.nan,'n_connected_components':len(comps),
            'gcc_connected_fraction':gcc/n_connected if n_connected else np.nan,'gcc_active_fraction':gcc/n_nodes if n_nodes else np.nan,
            'mean_degree_connected':2*n_edges/n_connected if n_connected else np.nan,'median_ppmi':float(np.median(weights)) if weights else np.nan,
            'effective_edge_count':eff_edges,'effective_edge_fraction':eff_edges/n_edges if n_edges else np.nan}

def build_detail(ids_tuple,mode,context,min_support):
    ids=list(ids_tuple); units_by_poem=context_dict(context); sub=primary_idx.loc[ids]; author_counts=Counter(sub.author_dir)
    node_mass=defaultdict(float); pair_mass=defaultdict(float); raw_support=defaultdict(int); total_mass=0.0
    for pid in ids:
        units=units_by_poem[pid]; U=len(units)
        if U==0: continue
        author=primary_idx.loc[pid].author_dir; unit_weight=1.0 if mode=='raw' else 1.0/(author_counts[author]*U)
        total_mass+=U*unit_weight
        for concepts in units:
            concepts=sorted(concepts)
            for u in concepts: node_mass[u]+=unit_weight
            for u,v in combinations(concepts,2):
                pair_mass[(u,v)]+=unit_weight; raw_support[(u,v)]+=1
    rows=[]
    for (u,v),support in raw_support.items():
        if support<int(min_support): continue
        pij=pair_mass[(u,v)]/total_mass; pi=node_mass[u]/total_mass; pj=node_mass[v]/total_mass
        if min(pij,pi,pj)<=0: continue
        ppmi=max(0.0,math.log2(pij/(pi*pj)))
        if ppmi>0: rows.append({'u':u,'v':v,'ppmi':ppmi,'raw_support':support,'weighted_support':pair_mass[(u,v)]})
    return pd.DataFrame(rows)


In [6]:
SEED=20260825; MC_DRAWS=1000
MAIN_WINDOWS=[(s,s+19) for s in range(1565,1606,5)]; assert len(MAIN_WINDOWS)==9
ids=primary.n_id.to_numpy(object); lo=primary.composition_min.to_numpy(int); hi=primary.composition_max.to_numpy(int)
rng=np.random.default_rng(SEED); sampled=np.empty((MC_DRAWS,len(primary)),dtype=int)
for j,(a,b) in enumerate(zip(lo,hi)): sampled[:,j]=a if a==b else rng.integers(a,b+1,size=MC_DRAWS)

records=[]
for m in range(MC_DRAWS):
    yrs=sampled[m]
    for start,end in MAIN_WINDOWS:
        selected=tuple(sorted(ids[(yrs>=start)&(yrs<=end)].tolist()))
        for candidate,spec in CANDIDATES.items():
            for mode in ('raw','author_balanced'):
                rec={'draw':m,'start':start,'end':end,'candidate':candidate,'mode':mode}
                rec.update(build_metrics(selected,mode,spec['context'],spec['min_support']))
                records.append(rec)
    if (m+1)%100==0: print('completed',m+1,'/',MC_DRAWS,'chronology draws')

mc=pd.DataFrame(records)
metric_cols=['n_poems','n_authors','effective_authors','top_author_share','n_context_units','n_nodes','n_connected_nodes',
             'connected_fraction','n_edges','density','n_connected_components','gcc_connected_fraction',
             'gcc_active_fraction','mean_degree_connected','median_ppmi','effective_edge_count','effective_edge_fraction']
summary_rows=[]
for (candidate,start,end,mode),grp in mc.groupby(['candidate','start','end','mode'],sort=True):
    row={'candidate':candidate,'start':start,'end':end,'mode':mode}
    for col in metric_cols:
        x=grp[col].astype(float)
        row[col+'_median']=float(x.median()); row[col+'_q10']=float(x.quantile(.10)); row[col+'_q90']=float(x.quantile(.90))
    summary_rows.append(row)
summary=pd.DataFrame(summary_rows)
summary['cell_pass']=(summary.connected_fraction_median.ge(MIN_CONNECTED_FRACTION) &
                      summary.gcc_connected_fraction_median.ge(MIN_GCC_CONNECTED_FRACTION) &
                      summary.mean_degree_connected_median.ge(MIN_MEAN_DEGREE_CONNECTED) &
                      summary.effective_edge_fraction_median.ge(MIN_EFFECTIVE_EDGE_FRACTION))
candidate_decision=(summary.groupby('candidate')
    .agg(pass_cells=('cell_pass','sum'),
         min_connected_fraction=('connected_fraction_median','min'),
         min_gcc_connected_fraction=('gcc_connected_fraction_median','min'),
         min_mean_degree_connected=('mean_degree_connected_median','min'),
         min_effective_edge_fraction=('effective_edge_fraction_median','min'),
         median_edges=('n_edges_median','median'),min_edges=('n_edges_median','min'),max_edges=('n_edges_median','max'))
    .reset_index())
candidate_decision['candidate_pass']=candidate_decision.pass_cells.eq(REQUIRED_PASS_CELLS)
order={k:v['selection_order'] for k,v in CANDIDATES.items()}
candidate_decision['selection_order']=candidate_decision.candidate.map(order)
candidate_decision=candidate_decision.sort_values('selection_order')
passing=candidate_decision[candidate_decision.candidate_pass].sort_values('selection_order')
SELECTED_CANDIDATE=passing.iloc[0].candidate if len(passing) else None

print('\nPHASE 11B CANDIDATE DECISION — STRUCTURE ONLY')
display(candidate_decision)
print('\nCell-level structural audit')
display(summary[['candidate','start','end','mode','n_nodes_median','n_connected_nodes_median','connected_fraction_median',
                 'n_edges_median','gcc_connected_fraction_median','mean_degree_connected_median',
                 'effective_edge_fraction_median','cell_pass']])
print('\nSELECTED CANDIDATE:',SELECTED_CANDIDATE if SELECTED_CANDIDATE else 'NO REPRESENTATION SELECTED')


completed 100 / 1000 chronology draws
completed 200 / 1000 chronology draws
completed 300 / 1000 chronology draws
completed 400 / 1000 chronology draws
completed 500 / 1000 chronology draws
completed 600 / 1000 chronology draws
completed 700 / 1000 chronology draws
completed 800 / 1000 chronology draws
completed 900 / 1000 chronology draws
completed 1000 / 1000 chronology draws

PHASE 11B CANDIDATE DECISION — STRUCTURE ONLY


,candidate,pass_cells,min_connected_fraction,min_gcc_connected_fraction,min_mean_degree_connected,min_effective_edge_fraction,median_edges,min_edges,max_edges,candidate_pass,selection_order
0,A_line_support2,0,0.035912,0.208333,1.142857,0.836847,14.0,8.0,31.0,False,1
1,B_line_support1,18,0.939203,0.841615,3.211180,0.910941,731.0,517.0,1100.0,True,2
2,C_token5_support1,18,1.000000,1.000000,11.629412,0.863345,2765.5,1977.0,4001.0,True,3



Cell-level structural audit


,candidate,start,end,mode,n_nodes_median,n_connected_nodes_median,connected_fraction_median,n_edges_median,gcc_connected_fraction_median,mean_degree_connected_median,effective_edge_fraction_median,cell_pass
0,A_line_support2,1565,1584,author_balanced,340.0,14.0,0.041176,8.0,0.285714,1.142857,0.963538,False
1,A_line_support2,1565,1584,raw,340.0,14.0,0.041176,8.0,0.285714,1.142857,0.968168,False
2,A_line_support2,1570,1589,author_balanced,362.0,20.0,0.055249,12.0,0.250000,1.200000,0.948034,False
3,A_line_support2,1570,1589,raw,362.0,20.0,0.055249,12.0,0.250000,1.200000,0.953159,False
4,A_line_support2,1575,1594,author_balanced,355.0,17.0,0.046575,10.0,0.294118,1.176471,0.936928,False
5,A_line_support2,1575,1594,raw,355.0,17.0,0.046575,10.0,0.294118,1.176471,0.973039,False
6,A_line_support2,1580,1599,author_balanced,408.0,24.0,0.058824,14.0,0.208333,1.166667,0.915401,False
7,A_line_support2,1580,1599,raw,408.0,24.0,0.058824,14.0,0.208333,1.166667,0.957535,False
8,A_line_support2,1585,1604,author_balanced,362.0,13.0,0.035912,9.0,0.538462,1.384615,0.836847,False
9,A_line_support2,1585,1604,raw,362.0,13.0,0.035912,9.0,0.538462,1.384615,0.868378,False



SELECTED CANDIDATE: B_line_support1


In [7]:
mid=((primary.composition_min+primary.composition_max)//2).to_numpy(int)
reference_rows=[]; reference_edges=[]
for start,end in MAIN_WINDOWS:
    selected=tuple(sorted(ids[(mid>=start)&(mid<=end)].tolist()))
    for candidate,spec in CANDIDATES.items():
        for mode in ('raw','author_balanced'):
            met=build_metrics(selected,mode,spec['context'],spec['min_support']).copy()
            met.update({'candidate':candidate,'start':start,'end':end,'mode':mode}); reference_rows.append(met)
            ed=build_detail(selected,mode,spec['context'],spec['min_support'])
            if len(ed):
                ed['candidate']=candidate; ed['start']=start; ed['end']=end; ed['mode']=mode; reference_edges.append(ed)
reference_metrics=pd.DataFrame(reference_rows)
reference_edges=pd.concat(reference_edges,ignore_index=True) if reference_edges else pd.DataFrame()

overlap=[]
if len(reference_edges):
    for candidate in CANDIDATES:
        for start,end in MAIN_WINDOWS:
            er=reference_edges[(reference_edges.candidate==candidate)&(reference_edges.start==start)&(reference_edges.end==end)&reference_edges['mode'].eq('raw')]
            eb=reference_edges[(reference_edges.candidate==candidate)&(reference_edges.start==start)&(reference_edges.end==end)&reference_edges['mode'].eq('author_balanced')]
            R=set(map(tuple,er[['u','v']].to_numpy())); B=set(map(tuple,eb[['u','v']].to_numpy())); union=R|B
            overlap.append({'candidate':candidate,'start':start,'end':end,'raw_edges':len(R),'balanced_edges':len(B),
                            'shared_edges':len(R&B),'edge_jaccard_raw_vs_balanced':len(R&B)/len(union) if union else np.nan})
weighting_overlap=pd.DataFrame(overlap)
print('MIDPOINT REFERENCE RAW-vs-BALANCED OVERLAP — QA ONLY')
display(weighting_overlap)


MIDPOINT REFERENCE RAW-vs-BALANCED OVERLAP — QA ONLY


,candidate,start,end,raw_edges,balanced_edges,shared_edges,edge_jaccard_raw_vs_balanced
0,A_line_support2,1565,1584,8,8,8,1.000000
1,A_line_support2,1570,1589,12,12,12,1.000000
2,A_line_support2,1575,1594,8,8,8,1.000000
3,A_line_support2,1580,1599,14,14,14,1.000000
4,A_line_support2,1585,1604,9,9,9,1.000000
5,A_line_support2,1590,1609,19,17,17,0.894737
6,A_line_support2,1595,1614,32,32,31,0.939394
7,A_line_support2,1600,1619,28,28,28,1.000000
8,A_line_support2,1605,1624,28,26,26,0.928571
9,B_line_support1,1565,1584,517,517,517,1.000000


In [8]:
OUT=Path('/content/gasr_phase11b_outputs'); OUT.mkdir(exist_ok=True)
mc.to_csv(OUT/'phase11b_mc_candidate_metrics.csv',index=False)
summary.to_csv(OUT/'phase11b_structural_summary.csv',index=False)
candidate_decision.to_csv(OUT/'phase11b_candidate_decision.csv',index=False)
reference_metrics.to_csv(OUT/'phase11b_reference_metrics.csv',index=False)
reference_edges.to_csv(OUT/'phase11b_reference_edges.csv',index=False)
weighting_overlap.to_csv(OUT/'phase11b_raw_vs_balanced_overlap.csv',index=False)
criteria.to_csv(OUT/'phase11b_frozen_structural_criteria.csv',index=False)

assert len(summary)==54,len(summary)
assert set(summary.candidate)==set(CANDIDATES)
assert set(summary['mode'])=={'raw','author_balanced'}
assert summary.n_nodes_median.gt(0).all() and summary.n_edges_median.gt(0).all()

print('\nPHASE 11B CHECKPOINT')
print('--------------------')
print('Primary chronology:',len(primary),'poems |',primary.author_dir.nunique(),'authors')
print('Main trajectory:',len(MAIN_WINDOWS),'windows | 20 years | step 5')
print('Chronology realizations:',MC_DRAWS)
print('Main vocabulary:',len(MAIN_VOCAB),'concepts')
print('Candidates evaluated:',', '.join(CANDIDATES))
print('Structural acceptance rule frozen before B/C results: TRUE')
print('Selected representation:',SELECTED_CANDIDATE if SELECTED_CANDIDATE else 'NONE')
print('Semantic temporal distance computed: FALSE')
print('Lexical-turnover vs persistent-concept rewiring computed: FALSE')
print('Change point computed: FALSE')
print('1580/1605 used for candidate selection: FALSE')
print('Historiographic labels used for candidate selection: FALSE')
print('If selected, next phase: preregister lexical turnover + persistent-concept rewiring statistics')
print('If none selected, next phase: representation contingency without historical tuning')
print('Outputs:',OUT)



PHASE 11B CHECKPOINT
--------------------
Primary chronology: 97 poems | 6 authors
Main trajectory: 9 windows | 20 years | step 5
Chronology realizations: 1000
Main vocabulary: 668 concepts
Candidates evaluated: A_line_support2, B_line_support1, C_token5_support1
Structural acceptance rule frozen before B/C results: TRUE
Selected representation: B_line_support1
Semantic temporal distance computed: FALSE
Lexical-turnover vs persistent-concept rewiring computed: FALSE
Change point computed: FALSE
1580/1605 used for candidate selection: FALSE
Historiographic labels used for candidate selection: FALSE
If selected, next phase: preregister lexical turnover + persistent-concept rewiring statistics
If none selected, next phase: representation contingency without historical tuning
Outputs: /content/gasr_phase11b_outputs
